# Lesson 4 — Spectral Analysis & Detector Noise

**Gravitational-Wave Data Analysis with Python**  
LVK Python Course — Module 4

> *“Noise is not the enemy — it is the canvas on which signals are painted.”*

---

## Table of Contents

1. [Introduction & Motivation](#1.-Introduction-&-Motivation)  
2. [The Discrete Fourier Transform (FFT)](#2.-The-Discrete-Fourier-Transform-(FFT))  
   - 2.1 Mathematical Foundation  
   - 2.2 The FFT Algorithm  
   - 2.3 Frequency Resolution and the Nyquist Limit  
   - 2.4 Window Functions & Spectral Leakage  
   - 2.5 Parseval's Theorem  
3. [Power Spectral Density (PSD)](#3.-Power-Spectral-Density-(PSD))  
   - 3.1 Definition and Physical Units  
   - 3.2 One-Sided vs Two-Sided PSD  
   - 3.3 Welch's Method  
   - 3.4 Median–Mean Method  
   - 3.5 Bias, Variance, and Resolution Trade-offs  
4. [Amplitude Spectral Density (ASD) & Sensitivity Curves](#4.-Amplitude-Spectral-Density-(ASD)-&-Sensitivity-Curves)  
   - 4.1 ASD Definition  
   - 4.2 LIGO/Virgo Sensitivity Curves: O1–O4  
   - 4.3 Noise Budget Anatomy  
5. [Q-Transform Spectrograms](#5.-Q-Transform-Spectrograms)  
   - 5.1 Theory: from STFT to Q-Transform  
   - 5.2 Key Parameters  
   - 5.3 Visualising GW Events  
6. [Identifying Glitches, Spectral Lines & Environmental Artefacts](#6.-Identifying-Glitches,-Spectral-Lines-&-Environmental-Artefacts)  
   - 6.1 Sources of Non-Gaussian Noise  
   - 6.2 Spectral Line Catalogue  
   - 6.3 Glitch Morphology & Gravity Spy  
   - 6.4 Environmental Monitoring & Auxiliary Channels  
   - 6.5 Data Quality Flags  
7. [Advanced Topics](#7.-Advanced-Topics)  
   - 7.1 Non-Stationarity and PSD Tracking  
   - 7.2 Cross-Spectral Density & Coherence  
   - 7.3 BayesLine Spectral Estimation  
8. [Student Exercises](#8.-Student-Exercises)  
9. [References](#9.-References)  


---
## 1. Introduction & Motivation

Gravitational-wave (GW) detectors are the most sensitive displacement sensors ever built. At their most sensitive frequencies (~100–200 Hz) the LIGO instruments can detect mirror displacements smaller than $10^{-19}$ m — a thousand times smaller than a proton. Yet, the raw output of each detector contains overwhelmingly more *noise* than signal.

Understanding the spectral properties of that noise is therefore **prerequisite** to every subsequent step of the analysis pipeline:

| Analysis step | Why spectral knowledge matters |
|---|---|
| Whitening | Divide by ASD to equalise frequency contributions |
| Matched filtering | SNR formula requires the one-sided PSD |
| Parameter estimation | Likelihood requires accurate noise model |
| Data quality | Identify and flag non-stationary noise |
| Detector characterisation | Diagnose noise sources to guide improvements |

This lesson covers the mathematical and practical tools needed to characterise detector noise in the frequency domain, starting from first principles and building up to the techniques used in real LVK analyses.

### Prerequisites
- Lesson 1 (GW detector basics)  
- Lesson 2 (Python for scientific computing)  
- Lesson 3 (Time-domain data handling)  
- Familiarity with complex numbers and basic signal processing  


In [ ]:
# Environment setup
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import signal
from scipy.fft import fft, ifft, fftfreq, rfft, rfftfreq

# Optional: install with  pip install gwpy gwosc
try:
    from gwpy.timeseries import TimeSeries
    from gwpy.frequencyseries import FrequencySeries
    from gwpy.segments import DataQualityFlag
    HAS_GWPY = True
except ImportError:
    HAS_GWPY = False
    print("gwpy not installed -- some cells will use synthetic data.")

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

rng = np.random.default_rng(seed=42)
print("Environment ready. NumPy:", np.__version__)


---
## 2. The Discrete Fourier Transform (FFT)

### 2.1 Mathematical Foundation

Given a discrete time series $x[n]$, $n = 0, 1, \ldots, N-1$, sampled at rate $f_s$ (sample interval $\Delta t = 1/f_s$), the **Discrete Fourier Transform** (DFT) is:

$$
X[k] = \sum_{n=0}^{N-1} x[n]\, e^{-i 2\pi k n / N}, \quad k = 0, 1, \ldots, N-1
$$

The corresponding **frequency bins** are:

$$
f_k = \frac{k}{N \Delta t} = \frac{k f_s}{N}, \quad k = 0, 1, \ldots, N-1
$$

The inverse DFT recovers the time series:

$$
x[n] = \frac{1}{N}\sum_{k=0}^{N-1} X[k]\, e^{i 2\pi k n / N}
$$

**Physical interpretation in the GW context:**  
Each complex coefficient $X[k]$ encodes the *amplitude* and *phase* of a sinusoidal component at frequency $f_k$. The GW strain $h(t)$ is a superposition of such components, and filtering in the Fourier domain allows us to weight each frequency by the inverse noise power — the basis of matched filtering (Lesson 5).

### 2.2 The FFT Algorithm

The naive DFT requires $\mathcal{O}(N^2)$ operations. The **Fast Fourier Transform** (FFT), introduced by Cooley & Tukey (1965), exploits the periodic structure of the complex exponentials to reduce this to $\mathcal{O}(N \log N)$. For a 4096-point array:

| Algorithm | Operations (approx.) |
|---|---|
| Naive DFT | 16 777 216 |
| FFT | 49 152 |
| Speed-up | x341 |

This speed-up is crucial for real-time GW searches where $N$ can exceed $10^6$.

### 2.3 Frequency Resolution and the Nyquist Limit

| Property | Formula | Typical LIGO value |
|---|---|---|
| **Nyquist frequency** | $f_{\rm Nyq} = f_s / 2$ | 2048 Hz (for $f_s = 4096$ Hz) |
| **Frequency resolution** | $\Delta f = f_s / N = 1 / T$ | 0.25 Hz for a 4-s segment |
| **Time resolution** | $\Delta t = 1 / f_s$ | 244 $\mu$s |

The **Nyquist–Shannon sampling theorem** states that a signal must be sampled at **at least twice** its highest frequency to be reconstructed without aliasing. GW data is low-pass filtered before downsampling to prevent aliased noise from contaminating the sensitive frequency band.

### 2.4 Window Functions & Spectral Leakage

When a segment of finite length is simply truncated (rectangular window), the discontinuities at the boundaries cause *spectral leakage*: power from strong spectral lines contaminates nearby frequency bins. Applying a **taper window** $w[n]$ before the FFT suppresses leakage:

$$
X_w[k] = \sum_{n=0}^{N-1} w[n]\, x[n]\, e^{-i 2\pi k n / N}
$$

Common windows in GW analysis:

| Window | Side-lobe level | FWHM (bins) | Use in GW |
|---|---|---|---|
| Rectangular | -13 dB | 0.89 | Rarely used alone |
| Hann | -31 dB | 1.50 | PSD estimation, whitening |
| Tukey (alpha=0.5) | -13 dB | ~1.0 | PSD, matched filtering |
| Kaiser (beta=10) | -80 dB | 2.1 | High-dynamic-range spectra |
| Flat-top | -93 dB | 3.77 | Amplitude calibration |

### 2.5 Parseval's Theorem

The DFT conserves signal energy:

$$
\sum_{n=0}^{N-1} |x[n]|^2 = \frac{1}{N} \sum_{k=0}^{N-1} |X[k]|^2
$$

The total power can therefore be computed equivalently in either domain — a useful sanity check when building analysis pipelines.


In [ ]:
# 2.3 Demonstrating frequency resolution
fs = 4096   # Hz
T  = 1.0    # duration (s)
N  = int(fs * T)
t  = np.arange(N) / fs

# Signal: two tones
x = np.sin(2 * np.pi * 150 * t) + 0.5 * np.sin(2 * np.pi * 300 * t)

freqs = rfftfreq(N, d=1/fs)
X     = rfft(x)
amp   = 2 * np.abs(X) / N

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(t[:200], x[:200])
axes[0].set(xlabel='Time (s)', ylabel='Amplitude', title='Time domain (first 50 ms)')

axes[1].semilogy(freqs, amp + 1e-10)
axes[1].axvline(150, color='C1', ls='--', label='150 Hz')
axes[1].axvline(300, color='C2', ls='--', label='300 Hz')
axes[1].set(xlabel='Frequency (Hz)', ylabel='Amplitude',
             title='FFT amplitude spectrum', xlim=(0, 500))
axes[1].legend()
plt.tight_layout()
plt.show()
print(f'Frequency resolution Df = {fs/N:.4f} Hz')
print(f'Nyquist frequency       = {fs//2} Hz')


In [ ]:
# 2.4 Window functions: time domain and frequency response
N_w = 1024
windows = {
    'Rectangular': np.ones(N_w),
    'Hann':        np.hanning(N_w),
    'Tukey a=0.5': signal.windows.tukey(N_w, alpha=0.5),
    'Kaiser b=10': np.kaiser(N_w, beta=10),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, w in windows.items():
    axes[0].plot(w, label=name, alpha=0.8)
    W    = np.abs(rfft(w, n=N_w * 8))
    W_dB = 20 * np.log10(W / W.max() + 1e-15)
    axes[1].plot(W_dB[:200], label=name, alpha=0.8)

axes[0].set(xlabel='Sample', ylabel='Amplitude', title='Window functions (time domain)')
axes[0].legend(fontsize=9)
axes[1].set(xlabel='Frequency bin (zero-padded)', ylabel='Magnitude (dB)',
             title='Window frequency response', ylim=(-120, 5))
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# 2.4 Spectral leakage: off-bin tone with different windows
fs_l = 1024
T_l  = 2.0
N_l  = int(fs_l * T_l)
t_l  = np.arange(N_l) / fs_l

f_line = 100.3   # Hz -- slightly off a grid bin -> leakage
x_l    = np.sin(2 * np.pi * f_line * t_l)
freqs_l = rfftfreq(N_l, d=1/fs_l)

fig, ax = plt.subplots(figsize=(12, 5))
for name, w in [('Rectangular', np.ones(N_l)),
                ('Hann', np.hanning(N_l)),
                ('Kaiser b=10', np.kaiser(N_l, beta=10))]:
    X_w   = rfft(x_l * w)
    scale = N_l * np.sqrt(np.mean(w**2))
    amp_w = 2 * np.abs(X_w) / scale
    ax.semilogy(freqs_l, amp_w + 1e-6, label=name, alpha=0.85)

ax.axvline(f_line, color='k', ls=':', alpha=0.5, label=f'True freq {f_line} Hz')
ax.set(xlabel='Frequency (Hz)', ylabel='Amplitude',
       title='Spectral leakage comparison', xlim=(90, 115), ylim=(1e-5, 3))
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 2.5 Parseval's theorem: numerical verification
x_p         = rng.standard_normal(4096)
X_p         = fft(x_p)
power_time  = np.sum(np.abs(x_p)**2)
power_freq  = np.sum(np.abs(X_p)**2) / len(x_p)
print('Parseval check:')
print(f'  Time-domain power  = {power_time:.6f}')
print(f'  Freq-domain power  = {power_freq:.6f}')
print(f'  Relative error     = {abs(power_time - power_freq)/power_time:.2e}')


---
## 3. Power Spectral Density (PSD)

### 3.1 Definition and Physical Units

The **Power Spectral Density** (PSD) describes how the power (variance) of a stationary random process is distributed across frequencies. For a continuous stationary process $x(t)$, the two-sided PSD is the Fourier transform of the autocorrelation function (Wiener–Khinchin theorem):

$$
S_{xx}(f) = \int_{-\infty}^{\infty} R_{xx}(\tau)\, e^{-i 2\pi f \tau}\, d\tau
$$

where $R_{xx}(\tau) = \langle x(t)\, x(t+\tau) \rangle$.

In the GW context the strain channel has units of **strain** (dimensionless), so the PSD has units of **strain$^2$/Hz** and the ASD has units of **strain/$\sqrt{\text{Hz}}$**.

### 3.2 One-Sided vs Two-Sided PSD

Because real-valued signals have conjugate-symmetric spectra ($S(-f) = S(f)$), it is conventional to work with the **one-sided PSD**:

$$
S_{1}(f) = 2\, S_{xx}(f), \quad f \geq 0
$$

so that integrating from 0 to $f_s/2$ gives the total variance:

$$
\sigma^2 = \int_0^{f_s/2} S_1(f)\, df
$$

Both `scipy.signal.welch` and `gwpy` return the one-sided PSD by default.

### 3.3 Welch's Method

A single-segment periodogram is a very *noisy* PSD estimate (its variance equals its mean squared — statistically useless). **Welch's method** reduces variance by averaging many overlapping periodograms:

1. Divide the data into $K$ overlapping segments of length $L$ (overlap typically 50%; apply a window function to each).  
2. Compute the periodogram of each windowed segment:
$$
\hat{P}_k(f) = \frac{|X_k(f)|^2}{f_s \sum_{n} w^2[n]}
$$
3. Average: $\hat{S}_{\rm Welch}(f) = \frac{1}{K}\sum_{k=1}^K \hat{P}_k(f)$

**Resolution–variance trade-off:**
- Longer segments: finer $\Delta f$, fewer averages, higher variance.  
- Shorter segments: coarser $\Delta f$, more averages, lower variance.  

In LIGO analyses the standard is **4-s segments** at $f_s = 4096$ Hz ($\Delta f = 0.25$ Hz), averaged over 32–256 s.

### 3.4 Median–Mean Method

Welch's arithmetic mean is sensitive to **outliers** (glitches). The **median–mean estimator** used in LALSuite/PyCBC replaces the per-frequency mean with a median, then rescales by $1/\ln 2$ (ratio mean/median for a $\chi^2_2$ distribution) to produce an asymptotically unbiased estimate:

$$
\hat{S}_{\rm mm}(f) = \frac{\operatorname{median}_k\{\hat{P}_k(f)\}}{\ln 2}
$$

This dramatically reduces the impact of loud short-duration glitches. A biased PSD directly biases matched-filter SNR, so robust estimation is critical.

### 3.5 Bias, Variance, and Resolution Trade-offs

| Estimator | Variance | Glitch bias | Freq. resolution |
|---|---|---|---|
| Periodogram (single) | $\propto S^2$ | high | $1/T$ |
| Welch ($K$ averages) | $\propto S^2/K$ | moderate | set by segment length |
| Median–mean | lower | **low** | same as Welch |
| DPSS/multitaper | low | low | $\sim K/T$ |

**Practical rule:** Always use the median–mean estimator (or gated Welch) when the data window contains non-Gaussian transients.


In [ ]:
# 3.3 Build coloured noise and Welch PSD
fs_w  = 4096
T_w   = 256  # seconds
N_w   = fs_w * T_w

# Analytic PSD model: flat above 100 Hz, rises as f^-2 below
freqs_full    = rfftfreq(N_w, d=1/fs_w)
freqs_full[0] = freqs_full[1]  # avoid DC division by zero
S_model       = 1e-46 * (1 + (100 / freqs_full)**2)
S_model[:5]   = S_model[5]

# Colour white noise
white_noise   = rng.standard_normal(N_w)
X_white       = rfft(white_noise)
X_coloured    = X_white * np.sqrt(S_model * fs_w / 2)
x_coloured    = np.real(np.fft.irfft(X_coloured, n=N_w))

# Welch PSD with different segment lengths
fig, ax = plt.subplots(figsize=(12, 5))
for seg_s in [1, 4, 16]:
    nperseg = fs_w * seg_s
    f_w, Pxx = signal.welch(x_coloured, fs=fs_w, nperseg=nperseg,
                             window='hann', noverlap=nperseg//2)
    ax.loglog(f_w[1:], Pxx[1:],
              label=f'Welch {seg_s}s segment (Df={1/seg_s:.3f} Hz)', alpha=0.85)

ax.loglog(freqs_full[1:], S_model[1:], 'k--', lw=2, label='True PSD (model)')
ax.set(xlabel='Frequency (Hz)', ylabel='PSD (strain$^2$/Hz)',
       title="Welch PSD -- segment length effect", xlim=(1, fs_w/2))
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 3.4 Median-mean estimator vs. Welch on glitchy data
x_glitchy = x_coloured.copy()
for _ in range(5):
    gi    = rng.integers(int(0.1 * N_w), int(0.9 * N_w))
    g_dur = int(0.05 * fs_w)  # 50 ms glitch
    x_glitchy[gi:gi + g_dur] += 200 * rng.standard_normal(g_dur)

nperseg_mm  = fs_w * 4
noverlap_mm = nperseg_mm // 2
window_mm   = np.hanning(nperseg_mm)

# Standard Welch
f_mm, Pxx_welch = signal.welch(x_glitchy, fs=fs_w, nperseg=nperseg_mm,
                                window='hann', noverlap=noverlap_mm)


def median_mean_psd(x, fs, nperseg, noverlap, window):
    """Median-mean PSD estimator (Finn 2001 / LALSuite convention)."""
    step   = nperseg - noverlap
    W_sum2 = np.sum(window**2)
    periogs = []
    for s in range(0, len(x) - nperseg + 1, step):
        Xk = rfft(x[s:s + nperseg] * window)
        P  = (2.0 / (fs * W_sum2)) * np.abs(Xk)**2
        P[0] /= 2
        P[-1] /= 2
        periogs.append(P)
    # Rescale median to be unbiased (chi^2_2: mean/median = 1/ln2)
    return rfftfreq(nperseg, d=1/fs), np.median(periogs, axis=0) / np.log(2)


f_mm2, Pxx_median = median_mean_psd(x_glitchy, fs_w, nperseg_mm, noverlap_mm, window_mm)

fig, ax = plt.subplots(figsize=(12, 5))
ax.loglog(f_mm[1:],  Pxx_welch[1:],  label='Welch (mean) -- glitchy data',  alpha=0.8)
ax.loglog(f_mm2[1:], Pxx_median[1:], label='Median-mean -- glitchy data',    alpha=0.8)
ax.loglog(freqs_full[1:], S_model[1:], 'k--', lw=2, label='True PSD')
ax.set(xlabel='Frequency (Hz)', ylabel='PSD (strain$^2$/Hz)',
       title='Welch vs. Median-Mean on glitchy data', xlim=(1, fs_w/2))
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 3.5 PSD variance reduction with number of averages
def estimate_psd_rel_std(x, fs, n_avg_list, target_f=150.0):
    nperseg_v = fs  # 1-s segments
    results = []
    for n_avg in n_avg_list:
        if n_avg * nperseg_v > len(x):
            results.append(np.nan)
            continue
        periogs = []
        for i in range(n_avg):
            seg = x[i*nperseg_v:(i+1)*nperseg_v] * np.hanning(nperseg_v)
            P   = (2.0 / (fs * np.sum(np.hanning(nperseg_v)**2))) * np.abs(rfft(seg))**2
            periogs.append(P)
        periogs = np.array(periogs)
        f_v = rfftfreq(nperseg_v, d=1/fs)
        col = periogs[:, np.argmin(np.abs(f_v - target_f))]
        results.append(np.std(col) / np.mean(col))
    return np.array(results)


n_avg_list = [1, 2, 4, 8, 16, 32, 64, 128]
rel_std    = estimate_psd_rel_std(x_coloured, fs_w, n_avg_list)
theory     = 1.0 / np.sqrt(np.array(n_avg_list, dtype=float))

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(n_avg_list, rel_std, 'o-', label='Estimated relative std')
ax.loglog(n_avg_list, theory,  '--', label=r'Theory: $1/\sqrt{K}$')
ax.set(xlabel='Number of averages K', ylabel='Relative std',
       title='PSD variance reduction with averaging')
ax.legend()
plt.tight_layout()
plt.show()


---
## 4. Amplitude Spectral Density (ASD) & Sensitivity Curves

### 4.1 ASD Definition

The **Amplitude Spectral Density** is the square root of the one-sided PSD:

$$
\tilde{h}(f) \equiv \sqrt{S_1(f)} \quad [\text{strain}/\sqrt{\text{Hz}}]
$$

The ASD is more convenient to plot because its range is compressed compared to the PSD: typical LIGO sensitivities span from $\sim 10^{-24}$ to $\sim 10^{-21}$ strain/$\sqrt{\text{Hz}}$, compared to $10^{-48}$–$10^{-42}$ strain$^2$/Hz for the PSD.

### 4.2 LIGO/Virgo Sensitivity Curves: O1–O4

The LVK detector network has improved dramatically between observing runs:

| Run | Period | LIGO BNS range | Key improvement |
|---|---|---|---|
| O1 | Sep 2015 – Jan 2016 | ~60–75 Mpc | First detection (GW150914) |
| O2 | Nov 2016 – Aug 2017 | ~80–100 Mpc | Virgo joined; GW170817 |
| O3a | Apr 2019 – Oct 2019 | ~120–130 Mpc | Squeezed light injection |
| O3b | Nov 2019 – Mar 2020 | ~130–150 Mpc | Improved mirror coatings |
| O4 | May 2023 – ongoing | ~160–190 Mpc | A+ upgrade (LIGO), AdV+ (Virgo) |

### 4.3 Noise Budget Anatomy

Characteristic ASD features:

- **Low-frequency wall** (below ~10–20 Hz): seismic noise and gravity gradients.  
- **Intermediate band** (20–200 Hz): thermal noise in mirror coatings and suspensions.  
- **High-frequency rise** (above ~200 Hz): quantum shot noise.  
- **Spectral lines**: violin modes (~500 Hz), mains hum (60/50 Hz), calibration lines.  

The total noise is the incoherent sum of independent contributions:

$$
S_{\rm total}(f) = S_{\rm seismic}(f) + S_{\rm thermal}(f) + S_{\rm shot}(f)
    + S_{\rm rad\,press}(f) + \cdots
$$

The **Standard Quantum Limit** (SQL) is the fundamental quantum noise floor without squeezing:

$$
h_{\rm SQL}(f) = \sqrt{\frac{4\hbar}{m (2\pi f)^2 L^2}}
$$

Frequency-dependent squeezing (injected from O3 onward) can surpass the SQL.


In [ ]:
# 4.2 Analytic sensitivity models O1-O4
f = np.logspace(np.log10(10), np.log10(4000), 2000)


def asd_model(f, seismic_knee=15, thermal_amp=1.0, shot_amp=1.0):
    """Simplified LIGO-like ASD model."""
    seismic = 1e-20 * (seismic_knee / np.maximum(f, 1e-3))**9
    thermal = thermal_amp * 3e-24 * (100 / np.maximum(f, 1))**0.5
    shot    = shot_amp    * 2e-24 * (f / 100)**1.5
    violin  = sum(5e-25 / (1 + ((f - fv) / 10)**2)**0.5 for fv in [500, 1000, 1500])
    return np.sqrt(seismic**2 + thermal**2 + shot**2) + violin


runs = {
    'O1 (2015)':   asd_model(f, 18, 1.6, 1.8),
    'O2 (2017)':   asd_model(f, 15, 1.2, 1.4),
    'O3 (2019)':   asd_model(f, 13, 1.0, 1.0),
    'O4 (2023)':   asd_model(f, 11, 0.85, 0.75),
    'Design (A+)': asd_model(f, 10, 0.7,  0.5),
}

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']
for (label, asd), color in zip(runs.items(), colors):
    ax.loglog(f, asd, label=label, color=color, lw=1.8)

ax.axvline(60,  color='gray', ls=':', alpha=0.6, lw=1)
ax.text(62, 3e-22, '60 Hz mains',  fontsize=8, color='gray')
ax.axvline(500, color='gray', ls=':', alpha=0.6, lw=1)
ax.text(510, 3e-22, 'violin\nmodes', fontsize=8, color='gray')

ax.set(xlabel='Frequency (Hz)', ylabel='ASD (strain/$\\sqrt{Hz}$)',
       title='LIGO sensitivity curves O1-O4 (analytic models)',
       xlim=(10, 4000), ylim=(5e-25, 5e-21))
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 4.3 Noise budget decomposition
fig, ax = plt.subplots(figsize=(12, 6))
f_nb = np.logspace(np.log10(10), np.log10(4000), 2000)

seismic   = 1e-20  * (13 / np.maximum(f_nb, 1e-3))**9
thermal   = 3e-24  * (100 / np.maximum(f_nb, 1))**0.5
shot      = 2e-24  * (f_nb / 100)**1.5
rad_press = 1e-24  * (30 / np.maximum(f_nb, 1))**2
total     = np.sqrt(seismic**2 + thermal**2 + shot**2 + rad_press**2)

m_mirror  = 40.0    # kg
L_arm     = 4000.0  # m
hbar      = 1.0546e-34
h_sql     = np.sqrt(4 * hbar / (m_mirror * (2*np.pi*f_nb)**2 * L_arm**2))

ax.loglog(f_nb, seismic,   ls='--', label='Seismic')
ax.loglog(f_nb, thermal,   ls='--', label='Thermal (coating + suspension)')
ax.loglog(f_nb, shot,      ls='--', label='Shot noise')
ax.loglog(f_nb, rad_press, ls='--', label='Radiation pressure')
ax.loglog(f_nb, total,     lw=2.5, color='black', label='Total')
ax.loglog(f_nb, h_sql,     lw=1.5, color='gray', ls=':', label='SQL')

ax.set(xlabel='Frequency (Hz)', ylabel='ASD (strain/$\\sqrt{Hz}$)',
       title='LIGO noise budget (O3 approximate)',
       xlim=(10, 4000), ylim=(1e-26, 1e-20))
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# 4.2 (bonus) Real LIGO ASD from GWOSC (requires gwpy)
if HAS_GWPY:
    try:
        gps_event = 1126259462  # GW150914
        print('Fetching H1 data around GW150914 from GWOSC...')
        h1 = TimeSeries.fetch_open_data('H1', gps_event - 32, gps_event + 32,
                                         verbose=False, cache=True)
        asd_real = h1.asd(fftlength=4, method='median')
        fig, ax = plt.subplots(figsize=(12, 5))
        asd_real.plot(ax=ax, label='H1 real ASD (O1, near GW150914)', color='steelblue')
        ax.set(xlabel='Frequency (Hz)', ylabel='ASD (strain/$\\sqrt{Hz}$)',
               title='Real LIGO-H1 ASD from GWOSC',
               xlim=(10, 2000), ylim=(5e-25, 1e-19))
        ax.legend()
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print(f'Could not fetch data: {exc}')
else:
    print('Install gwpy to load real LIGO data:  pip install gwpy')


---
## 5. Q-Transform Spectrograms

### 5.1 Theory: from STFT to Q-Transform

A **Short-Time Fourier Transform (STFT)** applies a fixed-length window that slides across the time series, producing a spectrogram with *constant* time–frequency resolution. This is suboptimal for chirping GW signals whose instantaneous frequency increases rapidly.

The **Q-Transform** (Brown 1991; Chatterji et al. 2004) uses windows whose length *scales inversely with frequency*, so that every time–frequency tile has a constant **quality factor**:

$$
Q = \frac{f_c}{\Delta f}
$$

where $f_c$ is the centre frequency and $\Delta f$ the bandwidth. At high $f$ the window is short (good time resolution); at low $f$ it is long (good frequency resolution). This tiling naturally matches the time–frequency evolution of a CBC chirp.

The Q-Transform of $x(t)$ at time $\tau$ and frequency $f_c$ is:

$$
X(\tau, f_c) = \int_{-\infty}^{\infty} x(t)\,
  w\!\left(t - \tau,\, f_c,\, Q\right)\, e^{-i 2\pi f_c t}\, dt
$$

where $w$ is a normalised Gaussian window of duration $\sigma_t = Q / (2\pi f_c \sqrt{2})$.

### 5.2 Key Parameters

| Parameter | Typical range | Effect |
|---|---|---|
| $Q$ | 4–100 | Lower Q: better time resolution; higher Q: better frequency resolution |
| `frange` | (10, 2000) Hz | Frequency range of the output |
| `tres` | 0.001–0.01 s | Output time-grid spacing |
| `norm` | `'median'` | ASD whitening method |

For BBH mergers: $Q \sim 8$–12. For BNS inspirals: $Q \sim 30$–40.

### 5.3 Visualising GW Events

The Q-spectrogram of a real GW event shows:
- A **frequency sweep** from low to high (the chirp).  
- The **merger** as a bright blob at the high-frequency end.  
- **Ringdown** as a fading oscillation after merger.  
- Background noise as diffuse speckling.  


In [ ]:
# 5.1 STFT spectrogram of a synthetic CBC-like chirp
from scipy.signal import chirp, spectrogram

fs_q = 4096
T_q  = 1.0
t_q  = np.linspace(0, T_q, int(fs_q * T_q), endpoint=False)

h_chirp = (chirp(t_q, f0=30, f1=500, t1=T_q, method='quadratic') *
           np.exp(-((t_q - T_q)**2) / (2 * 0.05**2)))

noise   = rng.standard_normal(len(t_q))
h_noisy = h_chirp / (np.std(noise) / 10.0) + noise  # SNR ~ 10

f_stft, t_stft, Sxx = spectrogram(h_noisy, fs=fs_q, nperseg=256,
                                    noverlap=240, window='hann')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(t_q, h_noisy, lw=0.5, alpha=0.7)
axes[0].plot(t_q, h_chirp, 'C1', lw=1.5, label='Clean chirp')
axes[0].set(xlabel='Time (s)', ylabel='Strain', title='Noisy chirp signal')
axes[0].legend()

im = axes[1].pcolormesh(t_stft, f_stft, 10*np.log10(Sxx + 1e-12),
                         cmap='viridis', shading='gouraud')
plt.colorbar(im, ax=axes[1], label='Power (dB)')
axes[1].set(xlabel='Time (s)', ylabel='Frequency (Hz)',
             title='STFT spectrogram (fixed resolution)', ylim=(0, 600))
plt.tight_layout()
plt.show()


In [ ]:
# 5.1 Simplified Q-Transform implementation
def qtransform(x, fs, Q=10, frange=(20, 800), n_f=100, n_t=200):
    """
    Simplified Q-Transform using Gaussian frequency-domain filters.
    Returns: times, freqs, normalised power map
    """
    N       = len(x)
    X       = rfft(x)
    f_fft   = rfftfreq(N, d=1/fs)
    f_fft[0] = f_fft[1]  # avoid DC singularity

    f_centers = np.logspace(np.log10(frange[0]), np.log10(frange[1]), n_f)
    times     = np.linspace(0, N / fs, n_t)
    power     = np.zeros((n_f, n_t))

    for i, fc in enumerate(f_centers):
        sigma_f  = fc / (Q * np.sqrt(2))
        H        = np.exp(-0.5 * ((f_fft - fc) / sigma_f)**2)
        xh       = np.fft.irfft(X * H, n=N)
        idx      = np.clip((times * fs).astype(int), 0, N - 1)
        power[i] = xh[idx]**2

    noise_floor = np.median(power, axis=1, keepdims=True)
    noise_floor[noise_floor == 0] = 1e-40
    return times, f_centers, power / noise_floor


times_q, freqs_q, power_q = qtransform(h_noisy, fs_q, Q=10,
                                         frange=(20, 600), n_f=120, n_t=400)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.pcolormesh(times_q, freqs_q, np.sqrt(power_q), cmap='RdYlBu_r',
                    norm=mcolors.PowerNorm(gamma=0.4, vmin=0.5, vmax=8),
                    shading='gouraud')
plt.colorbar(im, ax=ax, label='Normalised amplitude')
ax.set(xlabel='Time (s)', ylabel='Frequency (Hz)',
       title='Q-Transform spectrogram (Q=10)', yscale='log')
plt.tight_layout()
plt.show()


In [ ]:
# 5.2 Effect of Q value on time-frequency resolution
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, Q_val in zip(axes, [4, 12, 40]):
    _, fq, pq = qtransform(h_noisy, fs_q, Q=Q_val,
                            frange=(20, 600), n_f=120, n_t=400)
    im = ax.pcolormesh(times_q, fq, np.sqrt(pq), cmap='RdYlBu_r',
                        norm=mcolors.PowerNorm(gamma=0.4, vmin=0.5, vmax=8),
                        shading='gouraud')
    ax.set(xlabel='Time (s)', title=f'Q = {Q_val}', yscale='log')

axes[0].set_ylabel('Frequency (Hz)')
plt.suptitle('Q-Transform: effect of Q parameter', y=1.01)
plt.colorbar(im, ax=axes, label='Normalised amplitude')
plt.tight_layout()
plt.show()
print('Low Q -> fine time resolution, coarse frequency tracks.')
print('High Q -> fine frequency resolution, smeared time.')


In [ ]:
# 5.3 Real GW150914 Q-scan (requires gwpy)
if HAS_GWPY:
    try:
        gps_event = 1126259462
        print('Loading H1 data for GW150914 Q-scan...')
        h1_event = TimeSeries.fetch_open_data('H1',
                                               gps_event - 16, gps_event + 16,
                                               verbose=False, cache=True)
        h1_white = h1_event.whiten(4, 2)
        qgram = h1_white.q_transform(
                    outseg=(gps_event - 0.5, gps_event + 0.3),
                    qrange=(4, 64), frange=(20, 512))
        plot = qgram.plot(figsize=[8, 4])
        ax_q = plot.gca()
        ax_q.set_epoch(gps_event)
        ax_q.set(xlabel='Time (s) relative to GW150914',
                  ylabel='Frequency (Hz)',
                  title='H1 Q-scan: GW150914',
                  ylim=(20, 512), yscale='log')
        ax_q.colorbar(label='Normalised energy')
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print(f'Could not generate real Q-scan: {exc}')
else:
    print('Install gwpy:  pip install gwpy')


---
## 6. Identifying Glitches, Spectral Lines & Environmental Artefacts

Real detector data is **non-Gaussian** and **non-stationary**. Understanding the origin and morphology of non-astrophysical noise is critical for reliable event detection and parameter estimation.

### 6.1 Sources of Non-Gaussian Noise

| Category | Examples | Typical duration |
|---|---|---|
| Seismic | Earthquakes, microseism, anthropogenic | 1 s – hours |
| Acoustic | Airplanes, nearby roads | seconds – minutes |
| Electromagnetic | RF interference, power transients | < 1 ms – seconds |
| Optical | Light scattering, laser mode jumps | < 1 ms – seconds |
| Mechanical | Creak in the seismic isolation | < 100 ms |
| Cosmic rays | Charge deposition in optics | < 10 ms |
| Instrumental | ADC saturation, control noise | variable |

### 6.2 Spectral Line Catalogue

Persistent spectral lines arise from:

- **Power mains harmonics**: 60 Hz (US/Japan) or 50 Hz (Europe) and overtones.  
- **Calibration lines**: injected sinusoids to track the detector transfer function (e.g., LIGO-H1: ~34.7, ~36.7, ~1083.7 Hz).  
- **Violin modes**: mirror suspension fibre resonances (~500 Hz).  
- **Pendulum modes**: fundamental and harmonics (~0.6 Hz).  
- **Servo electronics**: feedback loops leaking into the output.  

These lines must be **notch-filtered** or excluded from the matched-filter frequency band.

### 6.3 Glitch Morphology & Gravity Spy

The [Gravity Spy](https://www.zooniverse.org/projects/zooniverse/gravity-spy) citizen science project (Zevin et al. 2017) classified O2 LIGO glitches into ~24 morphological classes:

| Glitch class | Typical duration | Freq. range | Notes |
|---|---|---|---|
| Blip | ~10 ms | 30–300 Hz | Unknown; possibly scattered light or cosmic rays |
| Koi Fish | ~100 ms | 30–500 Hz | Unknown origin |
| Scattered Light | 100 ms – 10 s | 10–100 Hz | Light bouncing off a moving surface |
| Tomte | ~10 ms | 20–200 Hz | Possible seismic coupling |
| Violin Mode Harmonic | narrowband | ~500 Hz | Mirror suspension modes |
| Power Line | persistent | 60, 120, 180 Hz | Electrical interference |

Machine learning classifiers (CNNs on Q-scan images) now automate classification in real-time.

### 6.4 Environmental Monitoring & Auxiliary Channels

Each LIGO detector records thousands of **auxiliary channels**: seismometers, accelerometers, microphones, magnetometers, laser power monitors, etc. **Coherence analysis** between the strain and an auxiliary channel reveals couplings:

$$
C_{xy}(f) = \frac{|S_{xy}(f)|^2}{S_{xx}(f)\, S_{yy}(f)} \in [0, 1]
$$

A coherence peak at 60 Hz between the strain and a magnetometer suggests electrical coupling to the power mains.

### 6.5 Data Quality Flags

| Category | Description | Action |
|---|---|---|
| Cat 1 | Hardware problems; detector not operating | Veto all events |
| Cat 2 | Known noise source coupling | Veto or warn |
| Cat 3 | Statistical correlations with aux. channels | Advisory |

Cat 2 vetoes remove ~1% of LIGO live time but dramatically reduce false-alarm rates. DQ flags for all public data are available at https://gwosc.org.


In [ ]:
# 6.2 Spectral line identification
fs_s = 4096
T_s  = 256
N_s  = fs_s * T_s
t_s  = np.arange(N_s) / fs_s

x_line = x_coloured[:N_s].copy()
for fl, al in zip([60, 120, 180, 500, 503, 506],
                   [5e-4, 2e-4, 1e-4, 8e-5, 6e-5, 4e-5]):
    x_line += al * np.sin(2 * np.pi * fl * t_s)

f_lp, Pxx_line = signal.welch(x_line, fs=fs_s, nperseg=fs_s*4,
                               window='hann', noverlap=fs_s*2)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].loglog(f_lp[1:], np.sqrt(Pxx_line[1:]))
for fl in [60, 120, 180, 500, 503, 506]:
    axes[0].axvline(fl, color='C1', ls='--', alpha=0.5, lw=0.8)
axes[0].set(xlabel='Frequency (Hz)', ylabel='ASD',
             title='ASD with injected spectral lines')

mask = (f_lp > 55) & (f_lp < 200)
axes[1].semilogy(f_lp[mask], np.sqrt(Pxx_line[mask]))
for fl in [60, 120, 180]:
    axes[1].axvline(fl, color='C1', ls='--', alpha=0.7, label=f'{fl} Hz')
axes[1].set(xlabel='Frequency (Hz)', ylabel='ASD',
             title='Zoom: power mains harmonics (60, 120, 180 Hz)')
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
# 6.3 Synthetic glitch zoo in the Q-Transform plane
fs_g   = 4096
T_g    = 2.0
t_g    = np.linspace(0, T_g, int(fs_g * T_g), endpoint=False)
noise_bg = rng.standard_normal(len(t_g)) * 3e-23


def make_blip(t, t0, amp=1.0, tau=0.01):
    return amp * np.exp(-0.5 * ((t - t0)/tau)**2) * np.sin(2*np.pi*150*(t - t0))


def make_scattered_light(t, t0, duration=0.5, fmax=80, fs=4096):
    mask   = (t > t0) & (t < t0 + duration)
    s      = np.zeros_like(t)
    f_inst = fmax * np.sin(np.pi * (t[mask] - t0) / duration)**2
    s[mask] = np.sin(2*np.pi * np.cumsum(f_inst) / fs)
    return s * np.exp(-((t - (t0 + duration/2))**2) / (2*(duration/4)**2))


signals = {
    'Blip (t=0.5s)':    make_blip(t_g, t0=0.5, amp=5e-22),
    'Scattered light':  make_scattered_light(t_g, t0=0.8, duration=0.8),
    'Power line 60 Hz': 2e-23 * np.sin(2 * np.pi * 60 * t_g),
}

fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
for ax, (label, sig) in zip(axes, signals.items()):
    _, fg, pg = qtransform(noise_bg + sig, fs_g, Q=10,
                            frange=(20, 500), n_f=100, n_t=300)
    im = ax.pcolormesh(np.linspace(0, T_g, 300), fg, np.sqrt(pg),
                        cmap='RdYlBu_r',
                        norm=mcolors.PowerNorm(gamma=0.5, vmin=0.5, vmax=6),
                        shading='gouraud')
    ax.set(xlabel='Time (s)', title=label, yscale='log')

axes[0].set_ylabel('Frequency (Hz)')
plt.suptitle('Synthetic glitch morphology in Q-Transform plane', y=1.01)
plt.colorbar(im, ax=axes, label='Normalised amplitude')
plt.tight_layout()
plt.show()


In [ ]:
# 6.4 Coherence between strain and an auxiliary channel
fs_c = 4096
T_c  = 64
N_c  = fs_c * T_c
t_c  = np.arange(N_c) / fs_c

mains_60   = np.sin(2 * np.pi * 60 * t_c)
strain_c   = rng.standard_normal(N_c) + 0.3 * mains_60   # strain channel
magnetic_c = rng.standard_normal(N_c) + mains_60          # magnetometer

f_coh, Cxy = signal.coherence(strain_c, magnetic_c, fs=fs_c,
                                nperseg=fs_c*4, noverlap=fs_c*2)

fig, ax = plt.subplots(figsize=(12, 4))
ax.semilogy(f_coh, Cxy)
ax.axvline(60,  color='C1', ls='--', label='60 Hz (injected coupling)')
ax.axvline(120, color='C2', ls='--', alpha=0.5, label='120 Hz')
ax.set(xlabel='Frequency (Hz)', ylabel='Coherence',
       title='Coherence: strain vs. magnetometer channel',
       xlim=(0, 200), ylim=(1e-4, 2))
ax.legend()
plt.tight_layout()
plt.show()
idx60 = np.argmin(np.abs(f_coh - 60))
print(f'Peak coherence at 60 Hz: {Cxy[idx60]:.4f}')


---
## 7. Advanced Topics

### 7.1 Non-Stationarity and PSD Tracking

Real LIGO data is **non-stationary**: the PSD evolves on timescales from seconds (control loop wandering) to hours (thermal drift). A single PSD estimated over 256 s may not accurately represent the noise during a candidate event.

Strategies:

- **Rolling PSD**: re-estimate in a sliding window, updating every few seconds (used by PyCBC and LALInference).  
- **Off-source PSD**: estimate from data *excluding* the $\pm 8$ s around the event, preventing signal bias.  
- **Gaussian process regression**: smooth function fitted to the median periodogram over many segments.  
- **Non-stationarity statistic**: ratio of expected to measured PSD variance flags elevated noise periods.

### 7.2 Cross-Spectral Density & Coherence

For two channels $x(t)$ and $y(t)$, the **cross-spectral density (CSD)** is:

$$
S_{xy}(f) = \langle X^*(f)\, Y(f) \rangle
$$

Applications:

- **Detector characterisation**: identify couplings between subsystems.  
- **Environmental noise subtraction**: regress out correlated noise.  
- **Stochastic background searches**: the GW background is coherent between spatially separated detectors; local noise is not (Allen & Romano 1999).  

### 7.3 BayesLine Spectral Estimation

**BayesLine** (Littenberg & Cornish 2015) simultaneously infers the GW signal parameters *and* the noise PSD, avoiding the circularity of estimating noise from data that may contain a signal.

The PSD is modelled as:
1. A smooth broadband component parameterised by a spline in log-frequency.  
2. A set of Lorentzian peaks representing spectral lines.  

Both are sampled in a nested-sampling/MCMC run together with the GW signal model. BayesLine is the standard noise model in **RIFT** and **LALInference**. Because the PSD is marginalised over, posterior distributions are properly broadened to account for noise model uncertainty.


In [ ]:
# 7.1 Rolling PSD tracks non-stationarity
fs_ns  = 4096
T_ns   = 128
N_ns   = fs_ns * T_ns
t_ns   = np.arange(N_ns) / fs_ns

envelope = 1.0 + 0.5 * np.sin(2 * np.pi * t_ns / T_ns)
x_ns     = envelope * rng.standard_normal(N_ns)

win_n      = fs_ns * 8
step_n     = fs_ns * 1
nperseg_ns = fs_ns * 2
target_f   = 200.0  # Hz

starts    = np.arange(0, N_ns - win_n, step_n)
times_ns  = starts / fs_ns + 4.0
psd_track = []

for s in starts:
    f_ns, P_ns = signal.welch(x_ns[s:s + win_n], fs=fs_ns,
                               nperseg=nperseg_ns, noverlap=nperseg_ns//2)
    psd_track.append(P_ns[np.argmin(np.abs(f_ns - target_f))])

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
axes[0].plot(t_ns[::100], envelope[::100])
axes[0].set(xlabel='Time (s)', ylabel='Noise envelope',
             title='Non-stationary noise: slow amplitude modulation')

axes[1].semilogy(times_ns, psd_track,
                  label=f'Rolling PSD @ {target_f:.0f} Hz (8 s window)')
axes[1].semilogy(times_ns, envelope[starts + win_n//2]**2,
                  'C1--', label='True noise power (envelope^2)')
axes[1].set(xlabel='Time (s)', ylabel='PSD',
             title='Rolling PSD tracks non-stationarity')
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
# 7.2 Inter-detector and detector-environmental coherence
fs_csd = 4096
T_csd  = 64
N_csd  = fs_csd * T_csd
t_csd  = np.arange(N_csd) / fs_csd

common_gw   = rng.standard_normal(N_csd) * 1e-23
env_seismic = rng.standard_normal(N_csd)*1e-22 + 0.1*np.sin(2*np.pi*1.0*t_csd)

h1_csd = rng.standard_normal(N_csd)*1e-23 + 0.1*common_gw + 0.05*env_seismic
l1_csd = rng.standard_normal(N_csd)*1e-23 + 0.1*common_gw + 0.02*env_seismic

f_cross, Cxy_HL = signal.coherence(h1_csd, l1_csd, fs=fs_csd,
                                    nperseg=fs_csd*4, noverlap=fs_csd*2)
f_cross, Cxy_He = signal.coherence(h1_csd, env_seismic, fs=fs_csd,
                                    nperseg=fs_csd*4, noverlap=fs_csd*2)

fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(f_cross, Cxy_HL, label='H1-L1 coherence (shared GW background)', alpha=0.8)
ax.semilogy(f_cross, Cxy_He, label='H1-Seismometer coherence', alpha=0.8)
ax.axhline(1/np.sqrt(T_csd/4), color='k', ls=':', label='~95% noise level')
ax.set(xlabel='Frequency (Hz)', ylabel='Coherence',
       title='Cross-spectral coherence: detector vs. environmental channel',
       xlim=(0.1, 100), xscale='log')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 7.3 BayesLine-inspired PSD model: smooth spline + Lorentzian lines
from scipy.interpolate import UnivariateSpline

f_bl = np.linspace(20, 1000, 4000)

psd_smooth = 3e-48 * (100 / np.maximum(f_bl, 1))**2 * (1 + (f_bl/400)**3)
for fl_b, A_b, g_b in [(60, 1e-46, 0.5), (120, 3e-47, 0.5), (500, 2e-46, 2.0)]:
    psd_smooth += A_b / (1 + ((f_bl - fl_b) / g_b)**2)
psd_meas = psd_smooth * np.exp(rng.standard_normal(len(f_bl)) * 0.15)

# Fit broadband spline in log space
log_f  = np.log(f_bl)
ctrl_i = np.linspace(0, len(f_bl)-1, 30, dtype=int)
spline = UnivariateSpline(log_f[ctrl_i], np.log(psd_meas[ctrl_i]),
                           k=3, s=len(ctrl_i)*0.05)
psd_fit = np.exp(spline(log_f))

fig, ax = plt.subplots(figsize=(12, 5))
ax.loglog(f_bl, psd_meas,   alpha=0.5, lw=0.8, label='Measured PSD (noisy)')
ax.loglog(f_bl, psd_smooth, 'k--',     lw=1.5, label='True PSD (with lines)')
ax.loglog(f_bl, psd_fit,    'C1',      lw=2,   label='BayesLine-like broadband fit')
ax.set(xlabel='Frequency (Hz)', ylabel='PSD (strain$^2$/Hz)',
       title='BayesLine approach: separate broadband and spectral line contributions')
ax.legend()
plt.tight_layout()
plt.show()
print('BayesLine marginalises over line positions and amplitudes via MCMC/nested sampling.')


---
## 8. Student Exercises

Exercises are ordered by increasing difficulty. Starred exercises (★) require real LIGO data via `gwpy`.

---

### 8.1 FFT Fundamentals

**Exercise 1.** A signal is sampled at $f_s = 2048$ Hz for $T = 10$ s.  
(a) What is the frequency resolution $\Delta f$?  
(b) What is the Nyquist frequency?  
(c) On which FFT bin does a 100.0 Hz sinusoid fall?  
(d) What happens if the sinusoid is at 100.1 Hz? Implement both cases and compare the periodograms with and without a Hann window.

**Exercise 2.** Verify Parseval's theorem numerically for a signal of three sinusoids with known amplitudes. Show that total power from the time domain equals the sum over DFT bins.

**Exercise 3.** Generate a linear chirp from 10 Hz to 300 Hz over 4 s (sampled at 4096 Hz). Apply rectangular, Hann, and Tukey ($\alpha=0.5$) windows before computing the FFT. Compare the amplitude spectra and discuss the leakage vs. amplitude accuracy trade-off.

---

### 8.2 PSD Estimation

**Exercise 4.** Generate 256 s of coloured Gaussian noise with PSD $S(f) = A (f_0/f)^\alpha$, $A = 10^{-45}$ strain$^2$/Hz, $f_0 = 100$ Hz, $\alpha = 2$. Estimate the PSD with Welch segment lengths of 1, 4, and 16 s. Discuss the bias–variance–resolution trade-off.

**Exercise 5.** Inject 10 short glitches (Gaussian pulses, width 20 ms) into the data from Exercise 4. Compare:  
(a) Standard Welch averaging;  
(b) Median–mean estimation;  
(c) Welch after gating (zeroing) the glitch-contaminated segments.  
Which method best recovers the true PSD?

**Exercise 6.** ★ Download 512 s of LIGO-H1 O3 data from GWOSC. Estimate the ASD with `TimeSeries.asd(fftlength=4, method='median')`. Identify at least five spectral lines and match them to their physical origin using the LIGO spectral line catalogue at https://www.gwosc.org.

---

### 8.3 Q-Transform Analysis

**Exercise 7.** Using the synthetic chirp from Section 5, generate Q-Transform spectrograms for $Q \in \{4, 8, 20, 60\}$. For each, measure the time and frequency resolution of the chirp track. Verify that $\Delta t \cdot \Delta f \approx Q / (2\pi)$ is approximately constant (time–frequency uncertainty principle for Gaussian windows).

**Exercise 8.** ★ Generate Q-scans for three GW events from GWOSC:  
(a) GW150914 (BBH merger);  
(b) GW170817 (BNS merger);  
(c) GW190521 (massive BBH candidate).  
Compare the time–frequency morphology. What $Q$ value best resolves each signal? Why?

---

### 8.4 Detector Characterisation

**Exercise 9.** ★ Download 1 hour of the LIGO-H1 microphone auxiliary channel `H1:PEM-EX_MIC_BS_OUT_DQ` from GWOSC. Compute the coherence with the strain channel. Identify frequencies with coherence $> 0.05$. What does acoustic coupling look like in the Q-plane?

**Exercise 10 (Open-ended).** The O3 spectral line catalogue lists lines at multiples of 331.9 Hz in H1.  
(a) Confirm these in a 1-hour ASD.  
(b) Use coherence with auxiliary channels to investigate the coupling source.  
(c) Estimate the effect on matched-filter SNR if the PSD is not accurately modelled at these frequencies.

---

### 8.5 Challenge Problems

**Exercise 11 (Advanced).** Implement a simple glitch detector inspired by Omicron:  
(a) Compute the Q-Transform of 60 s of data containing injected glitches.  
(b) For each time–frequency tile compute normalised energy $E = |X(\tau,f)|^2 / S(f)$.  
(c) Identify tiles with $E > 8$ as glitch candidates.  
(d) Cluster adjacent significant tiles. Compare to known injection times.

**Exercise 12 (Advanced).** Study PSD uncertainty on matched filtering:  
(a) Define a frequency-domain chirp template $h(f)$.  
(b) Compute matched-filter SNR $\rho = |\langle h, d \rangle|$ using three PSDs: true, short-window Welch, long-window Welch.  
(c) Compare expected vs. measured optimal SNR. Discuss implications for detection threshold setting.


---
## 9. References

### Textbooks and Reviews

1. **Jaranowski, P. & Krolak, A.** (2009)  
   *Data Analysis of Gravitational-Wave Signals from Spinning Neutron Stars.*  
   Cambridge University Press.

2. **Maggiore, M.** (2007)  
   *Gravitational Waves: Theory and Experiments.*  
   Oxford University Press. Chapters 7–9.

3. **Sathyaprakash, B.S. & Schutz, B.F.** (2009)  
   *Physics, Astrophysics and Cosmology with Gravitational Waves.*  
   Living Reviews in Relativity **12**, 2.  
   https://link.springer.com/article/10.12942/lrr-2009-2

4. **Oppenheim, A.V. & Schafer, R.W.** (2010)  
   *Discrete-Time Signal Processing*, 3rd ed. Prentice Hall.

5. **Percival, D.B. & Walden, A.T.** (1993)  
   *Spectral Analysis for Physical Applications.* Cambridge University Press.

### LVK Technical Documents

6. **LIGO Scientific Collaboration** (2015)  
   *Advanced LIGO.* CQG **32**, 074001.  
   https://doi.org/10.1088/0264-9381/32/7/074001

7. **Acernese, F. et al.** (2015)  
   *Advanced Virgo.* CQG **32**, 024001.  
   https://doi.org/10.1088/0264-9381/32/2/024001

8. **Buikema, A. et al.** (2020)  
   *Sensitivity and performance of Advanced LIGO in O3.*  
   Phys. Rev. D **102**, 062003.  
   https://doi.org/10.1103/PhysRevD.102.062003

### Algorithms

9. **Welch, P.D.** (1967)  
   *The use of fast Fourier transform for the estimation of power spectra.*  
   IEEE Trans. Audio Electroacoust. **AU-15**, 70–73.  
   https://doi.org/10.1109/TAU.1967.1161901

10. **Chatterji, S. et al.** (2004)  
    *Multiresolution techniques for the detection of GW bursts.*  
    CQG **21**, S1809.  
    https://doi.org/10.1088/0264-9381/21/20/024

11. **Littenberg, T.B. & Cornish, N.J.** (2015)  
    *Bayesian inference for spectral estimation of GW detector noise.*  
    Phys. Rev. D **91**, 084034.  
    https://doi.org/10.1103/PhysRevD.91.084034

12. **Zevin, M. et al.** (2017)  
    *Gravity Spy: Integrating Advanced LIGO Detector Characterization, ML, and Citizen Science.*  
    CQG **34**, 064003.  
    https://doi.org/10.1088/1361-6382/aa5cea

13. **Allen, B. & Romano, J.D.** (1999)  
    *Detecting a stochastic background of gravitational radiation.*  
    Phys. Rev. D **59**, 102001.  
    https://doi.org/10.1103/PhysRevD.59.102001

### Software & Data

14. **GWpy** — https://gwpy.github.io  
    Python package for GW data: `TimeSeries.asd()`, `q_transform()`, etc.

15. **GWOSC** — https://gwosc.org  
    Public detector data, DQ flags, spectral line catalogues.

16. **PyCBC** — https://pycbc.org  
    `pycbc.psd.welch`, `pycbc.psd.median_mean`, design sensitivity curves.

17. **SciPy signal processing** — https://docs.scipy.org/doc/scipy/reference/signal.html  
    `scipy.signal.welch`, `spectrogram`, `coherence`, window functions.

18. **LIGO DCC** — https://dcc.ligo.org  
    Technical documents; real noise budget curves (e.g., T1800044).

---

> **Next lesson:** Lesson 5 — Matched Filtering and Signal-to-Noise Ratio  
> *Building on the PSD tools developed here, we construct the optimal linear filter and compute the matched-filter SNR for compact binary coalescence signals.*
